# 04 - PostgreSQL dan SQL Analytics (FR-05)
**Prasyarat:** PostgreSQL aktif (lokal), `.env` terisi (salin dari `.env.example`), database `bank_campaign_db` sudah dibuat. Catatan: notebook ini paling mudah dijalankan di VS Code/Jupyter lokal, karena Colab tidak bisa mengakses PostgreSQL di komputer Anda.

- [ ] Jalankan `sql/schema.sql`
- [ ] Impor data bersih ke tabel `bank_campaign`
- [ ] Verifikasi jumlah baris = data sumber
- [ ] Jalankan query di `sql/campaign_analysis.sql` dan tulis interpretasi

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL          # perbaikan: import URL yang hilang
from sqlalchemy.exc import OperationalError

ROOT       = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_PATH = ROOT / "data" / "processed" / "bank_campaign_clean.csv"
SQL_DIR    = ROOT / "sql"
ENV_PATH   = ROOT / ".env"



In [2]:
load_dotenv(ENV_PATH, override=False)
keys = ["POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_HOST", "POSTGRES_PORT", "POSTGRES_DB"]
missing = [key for key in keys if not os.getenv(key)]
if missing:
    raise EnvironmentError(
        f"Environment variable belum diatur: {missing}. "
        "Buat file .env dari .env.example dan isi konfigurasi PostgreSQL."
    )

user, password, host, port, database = (os.getenv(key) for key in keys)
try:
    database_url = URL.create(
        "postgresql+psycopg",
        username=user,
        password=password,
        host=host,
        port=int(port),
        database=database,
    )
except (TypeError, ValueError) as error:
    raise ValueError("POSTGRES_PORT harus berupa nomor port yang valid.") from error

engine = create_engine(database_url, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        print(connection.execute(text("SELECT version()")).scalar())
except OperationalError as error:
    raise ConnectionError(
        "Koneksi PostgreSQL gagal. Pastikan service aktif, database tersedia, "
        "dan konfigurasi host/port/kredensial pada .env benar."
    ) from error

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [3]:
ddl = (SQL_DIR / "schema.sql").read_text(encoding="utf-8")
ddl_without_comments = "\n".join(
    line for line in ddl.splitlines() if not line.lstrip().startswith("--")
)
ddl_statements = [statement.strip() for statement in ddl_without_comments.split(";") if statement.strip()]
if not ddl_statements:
    raise ValueError("Tidak ada perintah DDL di sql/schema.sql")

with engine.begin() as connection:
    for statement in ddl_statements:
        connection.exec_driver_sql(statement)
print(f"Skema siap: {len(ddl_statements)} perintah DDL dijalankan.")

Skema siap: 6 perintah DDL dijalankan.


In [4]:
df = pd.read_csv(CLEAN_PATH)
print("Baris di CSV:", len(df))

# Tabel ini khusus untuk dataset proyek; muat ulang secara transaksional agar notebook aman diulang.
# Batch 1.000 baris membatasi jumlah parameter per perintah INSERT.
with engine.begin() as connection:
    connection.execute(text("DELETE FROM bank_campaign"))
    df.to_sql(
        "bank_campaign", connection, if_exists="append", index=False,
        method="multi", chunksize=1000,
    )
    n_db = connection.execute(text("SELECT COUNT(*) FROM bank_campaign")).scalar_one()
    if n_db != len(df):
        raise ValueError(f"Jumlah baris di PostgreSQL ({n_db}) tidak sama dengan CSV ({len(df)}).")
print("Baris di PostgreSQL:", n_db)

Baris di CSV: 45211


Baris di PostgreSQL: 45211


In [5]:
# Jalankan setiap query terpisah dan hentikan notebook jika ada query yang gagal.
sql_script = (SQL_DIR / "campaign_analysis.sql").read_text(encoding="utf-8")
import re
raw_blocks = re.split(r"(?=-- Q\d+\.)", sql_script)
query_blocks = [block.strip() for block in raw_blocks if re.search(r"-- Q\d+\.", block)]

with engine.connect() as connection:
    for block in query_blocks:
        lines = block.splitlines()
        title = next(line.strip() for line in lines if line.strip().startswith("-- Q"))
        sql_text = "\n".join(
            line for line in lines if line.strip() and not line.lstrip().startswith("--")
        ).strip().rstrip(";")
        if not sql_text:
            continue
        print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
        display(pd.read_sql(text(sql_text), connection))


-- Q0. Validasi jumlah baris (bandingkan dengan jumlah baris data sumber)


,total_rows
0,45211



-- Q1. KPI kampanye


,total_kontak,jumlah_yes,jumlah_no,response_rate_pct
0,45211,5289,39922,11.7



-- Q2. Response rate per pekerjaan (sertakan ukuran sampel)


...

,jumlah_kontak,total_kontak,response_rate_pct
0,1,17544,14.60
1,2,12505,11.20
2,3,5521,11.19
3,4,3522,9.00
4,5,1764,7.88
5,6,1291,7.13
6,7,735,6.39
7,8,540,5.93
8,9,327,6.42
9,10,266,5.26



-- Q7. Response rate berdasarkan hasil kampanye sebelumnya


,poutcome,total_kontak,response_rate_pct
0,success,1511,64.73
1,other,1840,16.68
2,failure,4901,12.61
3,unknown,36959,9.16



-- Q8. Response rate per bulan


,month,total_kontak,response_rate_pct
0,may,13766,6.72
1,jul,6895,9.09
2,aug,6247,11.01
3,jun,5341,10.22
4,nov,3970,10.15
5,apr,2932,19.68
6,feb,2649,16.65
7,jan,1403,10.12
8,oct,738,43.77
9,sep,579,46.46


## Interpretasi Hasil Query SQL

Query Q0–Q8 di `sql/campaign_analysis.sql` menjawab 8 dari key business questions PRD:

| Query | Business Question | Insight Kunci |
|---|---|---|
| Q0 | Validasi jumlah baris | 45.211 baris = sama dengan CSV sumber |
| Q1 | Tingkat keberhasilan kampanye | Response rate 11,70% (5.289 yes dari 45.211) |
| Q2 | Response rate per pekerjaan | `student` 28,68% dan `retired` 22,79% tertinggi |
| Q3 | Response rate per pendidikan | Tertiary 14,29% > secondary 10,77% > primary 8,53% |
| Q4 | Response rate per kelompok usia | Kelompok <25 dan 65+ cenderung lebih responsif |
| Q5 | Response rate per jenis kontak | `cellular` 14,92% vs `unknown` 4,07% |
| Q6 | Jumlah kontak vs response rate | 1 kontak punya rate tertinggi; makin banyak kontak rate turun |
| Q7 | Kampanye sebelumnya (poutcome) | `success` 64,73%; `failure` 14,29%; `unknown` 4,05% |
| Q8 | Response rate per bulan | `mar` 51,99% (n=477); `may` 6,72% (n=13.766) – perhatikan volume |

> **Catatan metodologis:** Seluruh angka bersifat deskriptif dan **bukan kausal**.
> Perbedaan response rate antar segmen bisa dipengaruhi oleh variabel lain yang tidak dikontrol.
> Validasi dengan eksperimen (A/B test) sebelum mengubah strategi kampanye.

### Query Data Quality
File `sql/data_quality.sql` menyediakan 10 query validasi (DQ-01 s.d. DQ-10) untuk:
- Verifikasi jumlah baris dan missing values
- Pengecekan duplikat, rentang nilai age/campaign/balance
- Distribusi nilai `pdays` dan proporsi `unknown` per kolom kategorikal


## PostgreSQL data quality checks

Run the DQ-01 to DQ-10 checks in `sql/data_quality.sql` against the loaded table. The checks cover row counts, nulls, duplicate records, target labels, ranges, and categorical values.

In [6]:
dq_script = (SQL_DIR / "data_quality.sql").read_text(encoding="utf-8")
dq_blocks = re.split(r"(?=-- DQ-\d+\.)", dq_script)
quality_results = {}

with engine.connect() as connection:
    for block in dq_blocks:
        title_match = re.search(r"-- (DQ-\d+\..*)", block)
        if not title_match:
            continue
        title = title_match.group(1)
        query = "\n".join(
            line for line in block.splitlines()
            if line.strip() and not line.lstrip().startswith("--")
        ).strip().rstrip(";")
        if not query:
            continue
        result = pd.read_sql(text(query), connection)
        quality_results[title.split(".")[0]] = result
        print(title)
        display(result)

checks = {
    "DQ-01 row count": int(quality_results["DQ-01"].iloc[0, 0]) == len(df),
    "DQ-02 no nulls": int(quality_results["DQ-02"].to_numpy().sum()) == 0,
    "DQ-03 no duplicate rows": quality_results["DQ-03"].empty,
    "DQ-04 valid target labels": quality_results["DQ-04"].empty,
    "DQ-05 valid age range": int(quality_results["DQ-05"]["usia_tidak_valid"].iloc[0]) == 0,
    "DQ-08 valid campaign count": int(quality_results["DQ-08"]["campaign_tidak_valid"].iloc[0]) == 0,
    "DQ-10 valid month labels": quality_results["DQ-10"].empty,
}
print("\nCore data quality checks:")
for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {check}")
if not all(checks.values()):
    raise ValueError("One or more core PostgreSQL data quality checks failed.")

DQ-01. Jumlah total baris (bandingkan dengan 45.211 dari CSV)

,total_rows
0,45211


DQ-02. Cek missing values per kolom


,null_age,null_job,null_marital,null_education,null_default,null_balance,null_housing,null_loan,null_contact,null_day,null_month,null_duration,null_campaign,null_pdays,null_previous,null_poutcome,null_y
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


DQ-03. Cek duplikat baris (tanpa kolom id)


...

DQ-09. Validasi saldo (balance) – tidak ada constraint range,


,balance_minimum,balance_maksimum,balance_rata_rata,balance_ekstrem_negatif
0,-8019,102127,1362.272058,0


DQ-10. Baris dengan nilai month di luar nama 12 bulan harus kosong.


,month,jumlah



Core data quality checks:
PASS - DQ-01 row count
PASS - DQ-02 no nulls
PASS - DQ-03 no duplicate rows
PASS - DQ-04 valid target labels
PASS - DQ-05 valid age range
PASS - DQ-08 valid campaign count
PASS - DQ-10 valid month labels
